# 04 - DFS（Depth-First Search，深度優先搜尋）

## 學習目標

完成本 Notebook 後，你將能夠：

- 使用鄰接串列表示圖（Graph）。
- 說明 DFS「一路走到底，再回頭」的搜尋方式。
- 使用遞迴與顯式堆疊實作 DFS。
- 理解 `visited` 集合的重要性。
- 分析 DFS 的時間與空間複雜度。


## 1. 圖與鄰接串列

圖由「節點（Vertex）」與「邊（Edge）」組成。以下無向圖包含這些連線：

```text
A-B, A-C, B-D, B-E, C-F, E-F
```

我們使用字典保存每個節點可以直接到達的鄰居。因為是無向圖，`A-B` 必須同時出現在 `A` 與 `B` 的鄰居清單中。


In [1]:
graph = {
    "A": ["B", "C"],
    "B": ["A", "D", "E"],
    "C": ["A", "F"],
    "D": ["B"],
    "E": ["B", "F"],
    "F": ["C", "E"],
}

for node, neighbors in graph.items():
    print(f"{node} 的鄰居：{neighbors}")


A 的鄰居：['B', 'C']
B 的鄰居：['A', 'D', 'E']
C 的鄰居：['A', 'F']
D 的鄰居：['B']
E 的鄰居：['B', 'F']
F 的鄰居：['C', 'E']


## 2. DFS 的核心想法

從起點開始：

1. 把目前節點標記為已拜訪。
2. 從目前節點選擇一個尚未拜訪的鄰居。
3. 對該鄰居重複相同過程，盡可能往深處走。
4. 如果沒有未拜訪的鄰居，就回到上一層，繼續嘗試其他方向。

這個「做完子問題後回到上一層」的動作稱為回溯（Backtracking）。

`visited` 非常重要：圖可能有環，例如 `A -> B -> A`。若不記錄已拜訪節點，搜尋會永遠繞圈。


In [2]:
def dfs_recursive(graph: dict[str, list[str]], start: str) -> list[str]:
    '''使用遞迴進行 DFS，回傳拜訪順序。'''
    visited = set()
    order = []

    def visit(node: str) -> None:
        # 一進入節點就立刻標記，避免之後重複進入。
        visited.add(node)
        order.append(node)

        # 依鄰接串列中的順序嘗試每個鄰居。
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visit(neighbor)

    if start in graph:
        visit(start)
    return order


order = dfs_recursive(graph, "A")
print("DFS 拜訪順序：", order)
assert order == ["A", "B", "D", "E", "F", "C"]


DFS 拜訪順序： ['A', 'B', 'D', 'E', 'F', 'C']


## 3. 逐步觀察遞迴與回溯

下方版本使用縮排顯示遞迴深度。看到「回到」時，代表該節點沒有其他未拜訪的鄰居，程式準備回到上一層。


In [3]:
def trace_dfs(graph: dict[str, list[str]], start: str) -> None:
    visited = set()

    def visit(node: str, depth: int) -> None:
        indent = "  " * depth
        print(f"{indent}進入 {node}")
        visited.add(node)

        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                print(f"{indent}從 {node} 前往 {neighbor}")
                visit(neighbor, depth + 1)

        print(f"{indent}完成 {node}，回到上一層")

    visit(start, 0)


trace_dfs(graph, "A")


進入 A
從 A 前往 B
  進入 B
  從 B 前往 D
    進入 D
    完成 D，回到上一層
  從 B 前往 E
    進入 E
    從 E 前往 F
      進入 F
      從 F 前往 C
        進入 C
        完成 C，回到上一層
      完成 F，回到上一層
    完成 E，回到上一層
  完成 B，回到上一層
完成 A，回到上一層


## 4. 使用堆疊改寫成迴圈

遞迴其實是由系統的函式呼叫堆疊協助記住「等一下要回去哪裡」。我們也可以自己建立 `stack`：

- 堆疊遵守 LIFO（Last In, First Out，後進先出）。
- `append()` 把節點放到堆疊頂端。
- `pop()` 取出最後放入的節點。

若希望迴圈版本與遞迴版本採用相同鄰居順序，放入堆疊時要使用 `reversed()`。因為最後放入的元素會最先取出。


In [4]:
def dfs_iterative(graph: dict[str, list[str]], start: str) -> list[str]:
    '''使用顯式堆疊進行 DFS，回傳拜訪順序。'''
    if start not in graph:
        return []

    visited = set()
    order = []
    stack = [start]

    while stack:
        node = stack.pop()

        # 同一節點可能由不同路線被放入堆疊，因此取出後仍要檢查。
        if node in visited:
            continue

        visited.add(node)
        order.append(node)

        # 反向放入，讓原清單中較前面的鄰居先被取出。
        for neighbor in reversed(graph.get(node, [])):
            if neighbor not in visited:
                stack.append(neighbor)

    return order


recursive_order = dfs_recursive(graph, "A")
iterative_order = dfs_iterative(graph, "A")
print("遞迴版本：", recursive_order)
print("迴圈版本：", iterative_order)
assert recursive_order == iterative_order


遞迴版本： ['A', 'B', 'D', 'E', 'F', 'C']
迴圈版本： ['A', 'B', 'D', 'E', 'F', 'C']


## 5. 遞迴深度與 `sys.setrecursionlimit`

如果圖像一條很長的鏈，遞迴 DFS 的深度可能接近節點數量，超過 Python 預設限制時會出現 `RecursionError`。

```python
import sys
sys.setrecursionlimit(20000)
```

調高上限有耗盡系統堆疊的風險，而且不會改善演算法複雜度。遇到非常深的圖時，顯式堆疊版本通常更安全。


In [5]:
import sys

print("目前的遞迴上限：", sys.getrecursionlimit())

# 教學示範：實務上應先確認輸入規模與系統資源。
sys.setrecursionlimit(20000)
print("調整後的遞迴上限：", sys.getrecursionlimit())


目前的遞迴上限： 3000
調整後的遞迴上限： 20000


## 6. 複雜度、應用與常見錯誤

使用鄰接串列時：

- 時間複雜度：$O(V+E)$，每個節點與每條邊最多檢查固定次數。
- 空間複雜度：$O(V)$，用於 `visited`、搜尋順序與遞迴／顯式堆疊。

常見應用：判斷能否到達、尋找連通元件、拓樸排序、偵測環、迷宮回溯。

常見錯誤：

- 忘記 `visited`，在有環的圖中無限循環。
- 太晚才標記已拜訪，造成同一節點被重複處理。
- 認為 DFS 一定能找到邊數最少的路徑；一般 DFS 不保證無權圖最短路徑。
- 遞迴版本沒有考慮輸入可能造成的遞迴深度。


## 7. 練習

實作 `can_reach_dfs(graph, start, goal)`，使用 DFS 判斷能否從 `start` 到達 `goal`。


In [6]:
def can_reach_dfs(
    graph: dict[str, list[str]], start: str, goal: str
) -> bool:
    # TODO：使用 visited 與 stack；找到 goal 時立刻回傳 True。
    pass


# 完成後可取消註解測試：
# assert can_reach_dfs(graph, "A", "F") is True
# assert can_reach_dfs(graph, "A", "Z") is False
